# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

-------

### Carga del corpus y construcción del índice

Se descargan los documentos, las consultas y los juicios de relevancia de NFCorpus. Se verifica que los IDs de los qrels existan en los datos cargados y se construye el índice invertido. La salida muestra el tamaño del vocabulario, la longitud media de los documentos y un ejemplo de tokenización.

In [1]:
from buscador import MODELS, SearchIndex, create_app, load_beir, tokenize

corpus = 'nfcorpus'
documents, queries, qrels = load_beir(corpus)
assert qrels.keys() <= queries.keys()
assert all(judgments.keys() <= documents.keys() for judgments in qrels.values())
index = SearchIndex(documents)

print(f'Corpus {corpus} | documentos {len(documents)} | vocabulario {len(index.vocabulary)}')
print(f'Longitud media {index.average_length:.2f} tokens')
print('Ejemplo de preprocesamiento', tokenize('The effects of processed foods on health'))

Corpus nfcorpus | documentos 3633 | vocabulario 19116
Longitud media 171.40 tokens
Ejemplo de preprocesamiento ['effect', 'process', 'food', 'health']


## TF-IDF con similitud coseno

La función tokeniza la consulta y calcula el peso de cada término del vocabulario con TF sublineal, 1 + log(tf), e IDF suavizada, log((N + 1) / (df + 1)) + 1. Recorre el índice invertido para acumular el producto escalar y divide por las normas de la consulta y del documento. Los empates se resuelven por ID.

In [2]:
from collections import Counter, defaultdict
from math import log, sqrt

def rank_tfidf(index, query, top_k=10):
    terms = Counter(tokenize(query))
    weights = {
        term: (1 + log(frequency)) * index.tfidf_idf[term]
        for term, frequency in terms.items()
        if term in index.vocabulary
    }
    query_norm = sqrt(sum(weight ** 2 for weight in weights.values()))
    scores = defaultdict(float)
    for term, query_weight in weights.items():
        for doc_id, frequency in index.postings[term].items():
            doc_weight = (1 + log(frequency)) * index.tfidf_idf[term]
            scores[doc_id] += query_weight * doc_weight
    for doc_id in scores:
        scores[doc_id] /= index.norms[doc_id] * query_norm
    return sorted(scores.items(), key=lambda item: (-item[1], item[0]))[:top_k]

example_query = 'dietary fiber and colon health'
print('TF-IDF', rank_tfidf(index, example_query, top_k=3))

TF-IDF [('MED-2578', 0.3178939674305644), ('MED-1234', 0.26856404875159173), ('MED-3136', 0.2569791647029023)]


## BM25

La función acumula la contribución de cada término distinto de la consulta. Usa k₁ = 1.2 y b = 0.75, pondera por la IDF de BM25 y ajusta la frecuencia del término por la longitud del documento respecto a la media. Devuelve los primeros resultados en orden de puntuación y resuelve los empates por ID.

In [3]:
def rank_bm25(index, query, top_k=10):
    terms = dict.fromkeys(tokenize(query))
    scores = defaultdict(float)
    k1, b = 1.2, 0.75
    for term in terms:
        if term not in index.vocabulary:
            continue
        for doc_id, frequency in index.postings[term].items():
            length_ratio = index.lengths[doc_id] / index.average_length
            denominator = frequency + k1 * (1 - b + b * length_ratio)
            scores[doc_id] += index.bm25_idf[term] * frequency * (k1 + 1) / denominator
    return sorted(scores.items(), key=lambda item: (-item[1], item[0]))[:top_k]

print('BM25', rank_bm25(index, example_query, top_k=3))

BM25 [('MED-2578', 13.94585406616142), ('MED-3136', 12.863075673728645), ('MED-2544', 12.814141294422726)]


## Evaluación con los qrels de test

Se incluyen todas las consultas que tienen juicios en `qrels/test.tsv`. La salida enumera sus identificadores y muestra el texto de tres ejemplos. Precision@10 divide los aciertos entre 10 y Recall@10 los divide entre todos los relevantes de la consulta. En ambas métricas se considera relevante una etiqueta mayor que cero. NDCG@10 usa el grado original como ganancia y el descuento logarítmico por posición. Se promedia cada métrica dando el mismo peso a todas las consultas.

### Cálculo de las métricas

Se ejecuta cada consulta de `test` con TF-IDF y BM25. Los primeros diez resultados se contrastan con los qrels para calcular Precision@10, Recall@10 y NDCG@10 por consulta. La salida muestra los IDs evaluados y el promedio de cada métrica por modelo. También se verifica que los rankings coincidan con los del motor que usa la API.

In [4]:
from math import log2
from statistics import mean

def metrics_at_10(ranking, judgments):
    grades = [judgments.get(doc_id, 0) for doc_id, _ in ranking[:10]]
    relevant = sum(grade > 0 for grade in judgments.values())
    hits = sum(grade > 0 for grade in grades)
    dcg = sum(grade / log2(rank + 1) for rank, grade in enumerate(grades, 1))
    ideal = sorted(judgments.values(), reverse=True)[:10]
    idcg = sum(grade / log2(rank + 1) for rank, grade in enumerate(ideal, 1))
    return hits / 10, hits / relevant if relevant else 0, dcg / idcg if idcg else 0

test_ids = sorted(qrels)
print(f'{len(test_ids)} consultas evaluadas')
print(', '.join(test_ids))
for qid in test_ids[:3]:
    print(qid, queries[qid])

metric_names = ('Precision@10', 'Recall@10', 'NDCG@10')
averages = {}
rankers = {'tfidf': rank_tfidf, 'bm25': rank_bm25}
for model, ranker in rankers.items():
    per_query = []
    for qid in test_ids:
        ranking = ranker(index, queries[qid], top_k=10)
        assert ranking == index.rank(queries[qid], model, 10)
        per_query.append(metrics_at_10(ranking, qrels[qid]))
    averages[model] = tuple(mean(values) for values in zip(*per_query))

print()
print(f"{'Modelo':<8}" + ''.join(f'{name:>16}' for name in metric_names))
for model, values in averages.items():
    print(f'{model:<8}' + ''.join(f'{value:>16.4f}' for value in values))

323 consultas evaluadas
PLAIN-1008, PLAIN-1018, PLAIN-102, PLAIN-1028, PLAIN-1039, PLAIN-1050, PLAIN-1066, PLAIN-1088, PLAIN-1098, PLAIN-1109, PLAIN-1119, PLAIN-112, PLAIN-1130, PLAIN-1141, PLAIN-1151, PLAIN-1161, PLAIN-1172, PLAIN-1183, PLAIN-1193, PLAIN-12, PLAIN-1203, PLAIN-1214, PLAIN-1225, PLAIN-123, PLAIN-1236, PLAIN-1249, PLAIN-1262, PLAIN-1275, PLAIN-1288, PLAIN-1299, PLAIN-1309, PLAIN-1320, PLAIN-133, PLAIN-1331, PLAIN-1342, PLAIN-1353, PLAIN-1363, PLAIN-1374, PLAIN-1387, PLAIN-1398, PLAIN-1409, PLAIN-1419, PLAIN-1429, PLAIN-143, PLAIN-1441, PLAIN-1453, PLAIN-1463, PLAIN-1473, PLAIN-1485, PLAIN-1496, PLAIN-1506, PLAIN-1516, PLAIN-1527, PLAIN-153, PLAIN-1537, PLAIN-1547, PLAIN-1557, PLAIN-1568, PLAIN-1579, PLAIN-1590, PLAIN-1601, PLAIN-1611, PLAIN-1621, PLAIN-1635, PLAIN-1645, PLAIN-165, PLAIN-1656, PLAIN-1667, PLAIN-1679, PLAIN-1690, PLAIN-1700, PLAIN-1710, PLAIN-1721, PLAIN-1731, PLAIN-1741, PLAIN-175, PLAIN-1752, PLAIN-1762, PLAIN-1772, PLAIN-1784, PLAIN-1794, PLAIN-1805, PL

### Comparación de los modelos

Se resta el promedio de TF-IDF al de BM25 para cada métrica. El signo identifica qué modelo obtuvo el valor más alto y la magnitud se expresa en puntos porcentuales. La comparación muestra en qué aspecto de la recuperación destaca cada modelo.

In [5]:
for name, tfidf, bm25 in zip(metric_names, averages['tfidf'], averages['bm25']):
    difference = (bm25 - tfidf) * 100
    if abs(difference) < 0.005:
        print(f'{name} registra un empate a la precisión mostrada.')
    else:
        winner = 'BM25' if difference > 0 else 'TF-IDF'
        print(f'{winner} supera al otro modelo por {abs(difference):.2f} puntos en {name}.')

print('Precision@10 mide cuántos de los primeros diez resultados son relevantes.')
print('Recall@10 indica cuánto del conjunto relevante se recuperó en esos diez lugares.')
print('NDCG@10 favorece los documentos de mayor relevancia en las primeras posiciones.')

BM25 supera al otro modelo por 0.59 puntos en Precision@10.
TF-IDF supera al otro modelo por 0.06 puntos en Recall@10.
BM25 supera al otro modelo por 0.49 puntos en NDCG@10.
Precision@10 mide cuántos de los primeros diez resultados son relevantes.
Recall@10 indica cuánto del conjunto relevante se recuperó en esos diez lugares.
NDCG@10 favorece los documentos de mayor relevancia en las primeras posiciones.


## Prueba de la API

Se inicia un servidor local temporal con el índice construido arriba. La consulta de prueba se define por separado de la evaluación. Se comprueban los dos modelos, los IDs, el orden, el límite solicitado y una solicitud inválida.

### Verificación de los endpoints

Se inicia un servidor temporal y se consulta `GET /health` para comprobar que NFCorpus esté cargado. Luego se envía la misma consulta a `POST /search` con ambos modelos y se verifican los IDs, el orden de las puntuaciones y el límite de tres resultados. Una solicitud inválida debe devolver HTTP 400.

In [6]:
import json
from threading import Thread
from urllib.error import HTTPError
from urllib.request import Request, urlopen
from wsgiref.simple_server import WSGIRequestHandler, make_server

class QuietHandler(WSGIRequestHandler):
    def log_message(self, format, *args):
        pass

with make_server('127.0.0.1', 0, create_app(index, corpus), handler_class=QuietHandler) as server:
    base_url = f'http://127.0.0.1:{server.server_port}'
    thread = Thread(target=server.serve_forever, daemon=True)
    thread.start()
    try:
        with urlopen(f'{base_url}/health', timeout=10) as response:
            health = json.load(response)
        assert health == {'status': 'ok', 'corpus': corpus, 'documents': len(documents)}
        print(health)
        for model in MODELS:
            body = {'query': 'dietary fiber and colon health', 'model': model, 'top_k': 3}
            request = Request(f'{base_url}/search', data=json.dumps(body).encode(),
                              headers={'Content-Type': 'application/json'}, method='POST')
            with urlopen(request, timeout=10) as response:
                payload = json.load(response)
            results = payload['results']
            assert payload['model'] == model and len(results) == 3
            ids = [row['document_id'] for row in results]
            scores = [row['score'] for row in results]
            assert len(set(ids)) == 3 and all(doc_id in documents for doc_id in ids)
            assert [row['rank'] for row in results] == [1, 2, 3]
            assert scores == sorted(scores, reverse=True)
            assert all(row['snippet'] and isinstance(row['title'], str) for row in results)
            print(model, list(zip(ids, [round(score, 4) for score in scores])))
        invalid = Request(f'{base_url}/search', data=b'{}', method='POST')
        try:
            urlopen(invalid, timeout=10).close()
        except HTTPError as error:
            assert error.code == 400
            print('Solicitud inválida rechazada con HTTP 400')
        else:
            raise AssertionError('Se esperaba HTTP 400')
    finally:
        server.shutdown()
        thread.join()

{'status': 'ok', 'corpus': 'nfcorpus', 'documents': 3633}
tfidf [('MED-2578', 0.3179), ('MED-1234', 0.2686), ('MED-3136', 0.257)]
bm25 [('MED-2578', 13.9459), ('MED-3136', 12.8631), ('MED-2544', 12.8141)]
Solicitud inválida rechazada con HTTP 400


## Publicación y prueba con Postman

![TF-IDF](tfidf.png)

![Health](health.png)

![BM25](bm25.png)